In [ ]:
import numpy

In [ ]:
!pip install ultralytics

In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import io
import random
from pathlib  import Path
import requests
from PIL import Image
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
print(torch.__version__)

2.11.0+cu128


In [ ]:
train_dir = Path(r"/content/drive/MyDrive/yolov8_train_car/Car_Demage_Severity/training")
test_dir = Path(r"/content/drive/MyDrive/yolov8_train_car/Car_Demage_Severity/validation")
save_dir = Path("/content/drive/MyDrive/yolov8_train_car/pkl")
save_dir.mkdir(parents=True, exist_ok=True)

model_path = save_dir / 'cnn_car.pkl'
seed=42
random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [ ]:
# Transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
weights = ResNet18_Weights.DEFAULT
eval_transform = weights.transforms()

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.03,
        )
    ], p=0.5),
    transforms.RandomAffine(
        degrees=7,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05),
    ),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=eval_transform.mean, std=eval_transform.std),
])

In [ ]:
full_train_for_train = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
full_train_for_eval = datasets.ImageFolder(root=str(train_dir), transform=eval_transform)
test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)
import numpy as np
targets = np.array(full_train_for_train.targets)
indices = np.arange(len(targets))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=seed,
    stratify=targets,
)

train_dataset = Subset(full_train_for_train, train_idx)
val_dataset = Subset(full_train_for_eval, val_idx)

class_names = full_train_for_train.classes
print("Classes:", class_names)
print(f"Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")

Classes: ['01-minor', '02-moderate', '03-severe']
Train/Val/Test: 2209/737/390


In [ ]:
# Khởi tạo DataLoader cho tập Train, Validation và Test
batch_size = 32  # Bạn có thể điều chỉnh kích thước batch tùy ý

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Đã khởi tạo các DataLoader thành công!")

Đã khởi tạo các DataLoader thành công!


In [ ]:
train_targets = targets[train_idx]
class_counts = np.bincount(train_targets, minlength=len(class_names))

class_weights = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05,
)

print("Class counts:", dict(zip(class_names, class_counts)))
print("Class weights:", class_weights)

Class counts: {'01-minor': np.int64(807), '02-moderate': np.int64(693), '03-severe': np.int64(709)}
Class weights: tensor([0.9124, 1.0625, 1.0386], device='cuda:0')


In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_true = []
    all_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)

            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": total_correct / max(total_samples, 1),
        "macro_f1": f1_score(all_true, all_pred, average="macro"),
        "y_true": all_true,
        "y_pred": all_pred,
    }


def train_one_phase(model, train_loader, val_loader, device, epochs, optimizer, save_path):
    history = []
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            running_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

        train_metrics = {
            'loss': running_loss / max(total_samples, 1),
            'accuracy': running_correct / max(total_samples, 1),
        }
        val_metrics = evaluate(model, val_loader, device)

        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            torch.save(model.state_dict(), save_path)

        history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
        print(
            f"Epoch {epoch:02d} | train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f}"
        )

    print(f'Best val_acc: {best_val_acc:.4f}')
    return history

In [ ]:


# Build transfer learning model
model = models.resnet18(weights=weights).to(device)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(num_ftrs, 3),
).to(device)

# Phase 1: freeze backbone, train only classifier head
for name, param in model.named_parameters():
    param.requires_grad = name.startswith('fc.')

optimizer_phase1 = torch.optim.AdamW(
    model.fc.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)
print('Phase 1: train classifier head')
history_phase1 = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=50,
    optimizer=optimizer_phase1,
    save_path=model_path,
)

Phase 1: train classifier head
Epoch 01 | train_loss=1.1107 train_acc=0.4341 | val_loss=0.0000 val_acc=0.5183
Epoch 02 | train_loss=0.9362 train_acc=0.5591 | val_loss=0.0000 val_acc=0.6472
Epoch 03 | train_loss=0.8626 train_acc=0.6188 | val_loss=0.0000 val_acc=0.6703
Epoch 04 | train_loss=0.8222 train_acc=0.6446 | val_loss=0.0000 val_acc=0.6961
Epoch 05 | train_loss=0.8040 train_acc=0.6514 | val_loss=0.0000 val_acc=0.6649
Epoch 06 | train_loss=0.7650 train_acc=0.6849 | val_loss=0.0000 val_acc=0.7096
Epoch 07 | train_loss=0.7510 train_acc=0.6840 | val_loss=0.0000 val_acc=0.7246
Epoch 08 | train_loss=0.7525 train_acc=0.6922 | val_loss=0.0000 val_acc=0.7178
Epoch 09 | train_loss=0.7435 train_acc=0.6935 | val_loss=0.0000 val_acc=0.7354
Epoch 10 | train_loss=0.7227 train_acc=0.7067 | val_loss=0.0000 val_acc=0.7436
Epoch 11 | train_loss=0.7277 train_acc=0.7184 | val_loss=0.0000 val_acc=0.7300
Epoch 12 | train_loss=0.7178 train_acc=0.7076 | val_loss=0.0000 val_acc=0.7395
Epoch 13 | train_loss

In [ ]:
# Phase 2: fine-tune layer4 + fc
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer_phase2 = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 5e-5},
], weight_decay=1e-4)
for param in model.fc.parameters():
    param.requires_grad = True

optimizer_phase2 = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 5e-4},
])
print('Phase 2: fine-tune layer4 + fc')
history_phase2 = train_one_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=50,
    optimizer=optimizer_phase2,
    save_path=model_path,
)

# Load best checkpoint and evaluate test
model.load_state_dict(torch.load(model_path, map_location=device))
test_metrics = evaluate(model, test_loader, device, criterion)

print("Test accuracy:", test_metrics["accuracy"])
print("Test macro_f1:", test_metrics["macro_f1"])

print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    target_names=class_names,
))

print(confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
))

Phase 2: fine-tune layer4 + fc
Epoch 01 | train_loss=0.6487 train_acc=0.7605 | val_loss=0.0000 val_acc=0.7910
Epoch 02 | train_loss=0.5181 train_acc=0.8393 | val_loss=0.0000 val_acc=0.8141
Epoch 03 | train_loss=0.4508 train_acc=0.8705 | val_loss=0.0000 val_acc=0.8209
Epoch 04 | train_loss=0.4015 train_acc=0.9027 | val_loss=0.0000 val_acc=0.7951
Epoch 05 | train_loss=0.3650 train_acc=0.9181 | val_loss=0.0000 val_acc=0.7978
Epoch 06 | train_loss=0.3185 train_acc=0.9457 | val_loss=0.0000 val_acc=0.7951
Epoch 07 | train_loss=0.2951 train_acc=0.9615 | val_loss=0.0000 val_acc=0.7870
Epoch 08 | train_loss=0.2842 train_acc=0.9660 | val_loss=0.0000 val_acc=0.8046
Epoch 09 | train_loss=0.2647 train_acc=0.9756 | val_loss=0.0000 val_acc=0.8046
Epoch 10 | train_loss=0.2595 train_acc=0.9774 | val_loss=0.0000 val_acc=0.8141
Epoch 11 | train_loss=0.2655 train_acc=0.9733 | val_loss=0.0000 val_acc=0.8087
Epoch 12 | train_loss=0.2496 train_acc=0.9851 | val_loss=0.0000 val_acc=0.7978
Epoch 13 | train_loss

In [ ]:
# Predict from image URL
def predict_from_url(url: str):
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()

    image = Image.open(io.BytesIO(resp.content)).convert('RGB')
    x = eval_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu()

    idx = int(torch.argmax(probs).item())
    return {
        'label': class_names[idx],
        'confidence': float(probs[idx].item()),
        'probs': {class_names[i]: float(probs[i].item()) for i in range(len(class_names))},
    }

In [ ]:
# Example URL prediction
import cv2
sample_url = 'https://tamanhhospital.vn/wp-content/uploads/2024/06/hinh-anh-nhan-biet-kien-ba-khoang.jpg'
result = predict_from_url(sample_url)
print(result)

{'label': '03-severe', 'confidence': 0.880948007106781, 'probs': {'01-minor': 0.07633774727582932, '02-moderate': 0.04271431639790535, '03-severe': 0.880948007106781}}
